# NegotiateEnv Long-Horizon Training

**Team**: Kushal Adhyaru & Mayuka Kothuru  
**Repository**: https://github.com/kushal511/saas-negotiation-env

Training configuration: 1000 episodes, 50 turns (long-horizon planning)

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/kushal511/saas-negotiation-env.git"
REPO_NAME = "saas-negotiation-env"

if os.path.exists(REPO_NAME):
    print(f"Repository exists. Pulling latest changes...")
    %cd {REPO_NAME}
    !git fetch origin
    !git reset --hard origin/main
    !git pull
else:
    print(f"Cloning repository...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}

print("Repository ready with latest code!")

## 3. Install Dependencies

In [ ]:
!pip install -q -e .
!pip install -q requests uvicorn fastapi psutil
print("Dependencies installed!")

## 4. Start Local Environment Server

In [ ]:
import subprocess
import time
import requests
import signal
import psutil

print("Starting environment server...")

# Kill any existing processes on port 7860
for proc in psutil.process_iter(['pid', 'name', 'connections']):
    try:
        for conn in proc.connections():
            if conn.laddr.port == 7860:
                print(f"Killing existing process on port 7860 (PID: {proc.pid})")
                proc.kill()
                time.sleep(1)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

time.sleep(2)

# Start server in background with minimal logging
server_log = open('/tmp/server.log', 'w')
server_process = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'negotiate_env.server.app:app', 
     '--host', '0.0.0.0', '--port', '7860', '--log-level', 'error'],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid  # Create new process group
)

# Wait for server with better error handling
ENV_URL = "http://localhost:7860"
server_ready = False

for attempt in range(20):
    try:
        time.sleep(2)
        response = requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
        if response.status_code == 200:
            print(f"Server ready at {ENV_URL}")
            server_ready = True
            break
    except requests.exceptions.ConnectionError:
        if attempt < 19:
            continue
    except Exception as e:
        if attempt < 19:
            continue

if not server_ready:
    print("Server failed to start. Check logs:")
    !tail -30 /tmp/server.log
    raise RuntimeError("Environment server did not start")

print(f"Environment URL: {ENV_URL}")

## 5. Install Training Dependencies

In [ ]:
!pip uninstall -y -q vllm 2>/dev/null || true
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.11.0" transformers accelerate peft datasets
print("Training dependencies installed!")

## 6. Expected Performance

In [ ]:
print("="*70)
print("EXPECTED PERFORMANCE IMPROVEMENT")
print("="*70)
print()
print("┌─────────────────────┬──────────────────┬──────────────────┬──────────────┐")
print("│ Metric              │ Baseline         │ After Training   │ Improvement  │")
print("├─────────────────────┼──────────────────┼──────────────────┼──────────────┤")
print("│ Avg Reward          │ 0.35 - 0.45      │ 0.45 - 0.55      │ +20-30%      │")
print("├─────────────────────┼──────────────────┼──────────────────┼──────────────┤")
print("│ Success Rate        │ ~60%             │ ~75-80%          │ +15-20%      │")
print("├─────────────────────┼──────────────────┼──────────────────┼──────────────┤")
print("│ Planning Depth      │ Shallow          │ Multi-step       │ Significant  │")
print("└─────────────────────┴──────────────────┴──────────────────┴──────────────┘")
print()
print("Why Performance Will Improve:")
print("1. Fixed Reward Scaling: -0.002/turn × 50 = -0.10 (fair!)")
print("2. More Training Data: 1000 episodes")
print("3. GRPO Algorithm: Optimized for sparse rewards")
print("4. Long-Horizon Features: 50 turns, 300+ instructions")
print("5. Session Management: Fixed API calls with session_id")
print("="*70)

## 7. Run Training

In [ ]:
print("Starting training (1000 episodes, 50 turns)...")
print()

!python train_negotiate_unsloth.py \
    --env-url http://localhost:7860 \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-long-horizon-output \
    --num-episodes 1000 \
    --max-turns 50

print("\nTraining complete!")
print("Model saved to: negotiate-long-horizon-output/")

## 8. Training Results

In [ ]:
import json
import os

print("="*60)
print("TRAINING RESULTS")
print("="*60)

trainer_state_path = "negotiate-long-horizon-output/trainer_state.json"

if os.path.exists(trainer_state_path):
    with open(trainer_state_path) as f:
        state = json.load(f)
    
    log_history = state.get("log_history", [])
    rewards = []
    for entry in log_history:
        reward = entry.get("env_reward") or entry.get("reward") or entry.get("train/env_reward") or entry.get("train/reward")
        if reward is not None:
            rewards.append(float(reward))
    
    if rewards:
        print(f"\nEpisodes: {len(rewards)}")
        print(f"Initial Reward: {rewards[0]:.4f}")
        print(f"Final Reward: {rewards[-1]:.4f}")
        print(f"Best Reward: {max(rewards):.4f}")
        print(f"Average Reward: {sum(rewards)/len(rewards):.4f}")
        
        improvement = rewards[-1] - rewards[0]
        improvement_pct = (improvement / abs(rewards[0]) * 100) if rewards[0] != 0 else 0
        print(f"\nImprovement: {improvement:+.4f} ({improvement_pct:+.1f}%)")
        
        print(f"\nExpected: 0.45 - 0.55")
        print(f"Actual: {rewards[-1]:.4f}")
        
        if rewards[-1] >= 0.45:
            print("\nTarget achieved!")
        elif rewards[-1] >= 0.40:
            print("\nGood progress, close to target.")
        else:
            print("\nNeeds more training.")
        
        print(f"\nTraining Milestones:")
        milestones = [0, len(rewards)//4, len(rewards)//2, 3*len(rewards)//4, len(rewards)-1]
        for idx in milestones:
            if idx < len(rewards):
                print(f"Episode {idx:4d}: {rewards[idx]:.4f}")
    else:
        print("\nNo reward data found")
else:
    print("\nTraining state not found")

print("\n" + "="*60)

## 9. Save Model to HuggingFace

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('Huggingface_Token')
    login(token=hf_token)
    print("Logged in to HuggingFace!")
except:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
output_dir = "negotiate-long-horizon-output"
repo_id = "KushalAdhyaru/negotiate-env-long-horizon-1000ep"

if os.path.exists(output_dir):
    print(f"Uploading to {repo_id}...")
    try:
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            repo_type="model",
        )
        print(f"\nModel uploaded!")
        print(f"View at: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"Upload failed: {e}")
else:
    print("Output directory not found")

## Summary

**Statement 2: Super Long-Horizon Planning**

Implemented Features:
- Extended episodes (50 turns)
- 300+ scattered instructions
- 8-stage sales workflow
- Multi-deal negotiation
- Sparse rewards
- Proportional turn penalty

**Team**: Kushal Adhyaru & Mayuka Kothuru  
**Repository**: https://github.com/kushal511/saas-negotiation-env